In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import stim
import numpy as np
import pymatching

from qecsim.surface_code_rotated.circuit import rotated_surface_code

In [4]:
noise = 0.007
distance = 7

c_0_z = rotated_surface_code(distance = distance, rounds = distance, log_obs = "Z", state_init = "0", noise_after_clifford_depol = noise, noise_after_reset = noise, noise_depol_data_init = noise, noise_measure_flip = noise)
c_1_z = rotated_surface_code(distance = distance, rounds = distance, log_obs = "Z", state_init = "1", noise_after_clifford_depol = noise, noise_after_reset = noise, noise_depol_data_init = noise, noise_measure_flip = noise)
c_p_x = rotated_surface_code(distance = distance, rounds = distance, log_obs = "X", state_init = "+", noise_after_clifford_depol = noise, noise_after_reset = noise, noise_depol_data_init = noise, noise_measure_flip = noise)
c_m_x = rotated_surface_code(distance = distance, rounds = distance, log_obs = "X", state_init = "-", noise_after_clifford_depol = noise, noise_after_reset = noise, noise_depol_data_init = noise, noise_measure_flip = noise)
c_0_x, rec_pos_log_x = rotated_surface_code(distance = distance, rounds = distance, log_obs = "X", state_init = "0", noise_after_clifford_depol = noise, noise_after_reset = noise, noise_depol_data_init = noise, noise_measure_flip = noise)
c_1_x, rec_pos_log_x = rotated_surface_code(distance = distance, rounds = distance, log_obs = "X", state_init = "1", noise_after_clifford_depol = noise, noise_after_reset = noise, noise_depol_data_init = noise, noise_measure_flip = noise)
c_p_z, rec_pos_log_z = rotated_surface_code(distance = distance, rounds = distance, log_obs = "Z", state_init = "+", noise_after_clifford_depol = noise, noise_after_reset = noise, noise_depol_data_init = noise, noise_measure_flip = noise)
c_m_z, rec_pos_log_z = rotated_surface_code(distance = distance, rounds = distance, log_obs = "Z", state_init = "-", noise_after_clifford_depol = noise, noise_after_reset = noise, noise_depol_data_init = noise, noise_measure_flip = noise)

In [5]:
def building_ptm(circuit_0_z: stim.Circuit, circuit_1_z: stim.Circuit, 
                 circuit_0_x: stim.Circuit, circuit_1_x: stim.Circuit, 
                 circuit_p_x: stim.Circuit, circuit_m_x: stim.Circuit, 
                 circuit_p_z: stim.Circuit, circuit_m_z: stim.Circuit,
                 rec_pos_log_x : list, rec_pos_log_z : list, samples : int = 50) -> np.ndarray:

    """
    Building PTM by building each possible memory circuit and measurement combination
    -> + logical -> Measuring in x/z/y basis
    -> - logical -> Measuring in x/z/y basis
    -> 0 logical -> Measuring in x/z/y basis
    -> 1 logical -> Measuring in x/z/y basis
    -> +i logical -> Measuring in x/z/y basis

    What we do:
        * Determine the noise clean expectation of the measurement
        * Compute the noisy current observable status (xor noiseless initlial state by obs state)
        * Run QEC and check whether the noisy observable should be flipped or not
        * Flip the observable i.e. the clean expectation value
        * Build up PTM out of the expectaition values
     """

    #####################################
    # Builing Diagonal Entries of the PTM
    #####################################

    # Correct nosieless Measurements
    clean_meas_0_z = False
    clean_meas_1_z = True
    clean_meas_p_x = False
    clean_meas_m_x = True

    # Build a graphlike DEM and a matcher
    dem_0_z = circuit_0_z.detector_error_model(decompose_errors=True)
    m_0_z = pymatching.Matching.from_detector_error_model(dem_0_z)
    dem_1_z = circuit_1_z.detector_error_model(decompose_errors=True)
    m_1_z = pymatching.Matching.from_detector_error_model(dem_1_z)
    dem_p_x = circuit_p_x.detector_error_model(decompose_errors=True)
    m_p_x = pymatching.Matching.from_detector_error_model(dem_p_x)
    dem_m_x = circuit_m_x.detector_error_model(decompose_errors=True)
    m_m_x = pymatching.Matching.from_detector_error_model(dem_m_x)

    # Take m shots of detection events
    sampler_0_z = circuit_0_z.compile_detector_sampler()
    dets_0_z, obs_0_z = sampler_0_z.sample(samples, separate_observables=True)
    
    sampler_1_z = circuit_1_z.compile_detector_sampler()
    dets_1_z, obs_1_z = sampler_1_z.sample(samples, separate_observables=True)

    sampler_p_x = circuit_p_x.compile_detector_sampler()
    dets_p_x, obs_p_x = sampler_p_x.sample(samples, separate_observables=True)
    
    sampler_m_x = circuit_m_x.compile_detector_sampler()
    dets_m_x, obs_m_x = sampler_m_x.sample(samples, separate_observables=True)

    # Determine current noisy Operator states (i.e. xor obs from det sample with noiseless Measurement outcome)
    """
    THIS IS TOO MUCH WORK AND CAN BE SIMPLIFIED
    -> Just return freom each circuit the log rec tragets not only for the non determinstic emasurements
    -> Do analogue method than for the off diagonals
    -> This works but is more complicated and results in two methods beeing used in one function which can be mitigated
    """

    noisy_meas_0_z = []
    noisy_meas_1_z = []
    noisy_meas_p_x = []
    noisy_meas_m_x = []

    for curr_shot in range(samples):
        noisy_meas_0_z.append(obs_0_z[curr_shot,0] ^ clean_meas_0_z)
        noisy_meas_1_z.append(obs_1_z[curr_shot,0] ^ clean_meas_1_z)
        noisy_meas_p_x.append(obs_p_x[curr_shot,0] ^ clean_meas_p_x)
        noisy_meas_m_x.append(obs_m_x[curr_shot,0] ^ clean_meas_m_x)

    # Decode all sample round in one Batch decode
    pred_0_z = m_0_z.decode_batch(dets_0_z)
    pred_1_z = m_1_z.decode_batch(dets_1_z)
    pred_p_x = m_p_x.decode_batch(dets_p_x)
    pred_m_x = m_m_x.decode_batch(dets_m_x)

    # XOR flip with noiseless Measurement
    # -> Taking first entry for first logical observable
    final_meas_0_z = []
    final_meas_1_z = []
    final_meas_p_x = []
    final_meas_m_x = []

    for curr_shot in range(samples):
        final_meas_0_z.append( 1 - 2 * (pred_0_z[curr_shot,0].astype(np.int8) ^ np.int8(noisy_meas_0_z[curr_shot])))
        final_meas_1_z.append( 1 - 2 * (pred_1_z[curr_shot,0].astype(np.int8) ^ np.int8(noisy_meas_1_z[curr_shot])))
        final_meas_p_x.append( 1 - 2 * (pred_p_x[curr_shot,0].astype(np.int8) ^ np.int8(noisy_meas_p_x[curr_shot])))
        final_meas_m_x.append( 1 - 2 * (pred_m_x[curr_shot,0].astype(np.int8) ^ np.int8(noisy_meas_m_x[curr_shot])))

    # Take the Average of each of them
    mu_z_pz = np.average(final_meas_0_z)
    mu_z_mz = np.average(final_meas_1_z)
    mu_x_px = np.average(final_meas_p_x)
    mu_x_mx = np.average(final_meas_m_x)

    # Calc PTM entries
    r_zz = 1/2 * (mu_z_pz - mu_z_mz)
    r_xx = 1/2 * (mu_x_px - mu_x_mx)

    ######################################
    # Building Off-Diagonals of the Matrix
    ######################################

    """
    For the Off-Diagonals we can't directly use the DEM as we need the raw measurements to infer what logical state we have

    -> We use compile sampler to infer the logical state
    -> Convert into a dem to run the matching
    """

    # Build the normal measurement smaples and sample n shots
    smpl_0_x = circuit_0_x.compile_sampler()
    rstls_smpls_0_x = smpl_0_x.sample(shots = samples)
    smpl_1_x = circuit_1_x.compile_sampler()
    rstls_smpls_1_x = smpl_1_x.sample(shots = samples)
    smpl_p_z = circuit_p_z.compile_sampler()
    rstls_smpls_p_z = smpl_p_z.sample(shots = samples)
    smpl_m_z = circuit_m_z.compile_sampler()
    rstls_smpls_m_z = smpl_m_z.sample(shots = samples)

    # Getting the individual logical states after measurement
    log_state_0_x = []
    log_state_1_x = []
    log_state_p_z = []
    log_state_m_z = []

    for curr_shot in range(samples):

        measurement_res_0_x = -99
        measurement_res_1_x = -99
        measurement_res_p_z = -99
        measurement_res_m_z = -99

        # Do the first loop for all x basis measurements
        for current_tar_rec in rec_pos_log_x:
            
            # If this is the first, set the resutl to this measurement outcome
            if measurement_res_0_x == -99:
                measurement_res_0_x = rstls_smpls_0_x[curr_shot, current_tar_rec]

            # else xor measurement to the already present ones
            else:
                measurement_res_0_x ^= rstls_smpls_0_x[curr_shot, current_tar_rec]

            # If this is the first, set the resutl to this measurement outcome
            if measurement_res_1_x == -99:
                measurement_res_1_x = rstls_smpls_1_x[curr_shot, current_tar_rec]

            # else xor measurement to the already present ones
            else:
                measurement_res_1_x ^= rstls_smpls_1_x[curr_shot, current_tar_rec]

        # Do the same for z
        for current_tar_rec in rec_pos_log_z:
            
            # If this is the first, set the resutl to this measurement outcome
            if measurement_res_p_z == -99:
                measurement_res_p_z = rstls_smpls_p_z[curr_shot, current_tar_rec]

            # else xor measurement to the already present ones
            else:
                measurement_res_p_z ^= rstls_smpls_p_z[curr_shot, current_tar_rec]

            # If this is the first, set the resutl to this measurement outcome
            if measurement_res_m_z == -99:
                measurement_res_m_z = rstls_smpls_m_z[curr_shot, current_tar_rec]

            # else xor measurement to the already present ones
            else:
                measurement_res_m_z ^= rstls_smpls_m_z[curr_shot, current_tar_rec]

        # Appending the current measurements to the list
        log_state_0_x.append(measurement_res_0_x)
        log_state_1_x.append(measurement_res_1_x)
        log_state_p_z.append(measurement_res_p_z)
        log_state_m_z.append(measurement_res_m_z)

    # Build a graphlike DEM and a matcher
    dem_0_x = circuit_0_x.detector_error_model(decompose_errors=True)
    m_0_x = pymatching.Matching.from_detector_error_model(dem_0_x)
    dem_1_x = circuit_1_x.detector_error_model(decompose_errors=True)
    m_1_x = pymatching.Matching.from_detector_error_model(dem_1_x)
    dem_p_z = circuit_p_z.detector_error_model(decompose_errors=True)
    m_p_z = pymatching.Matching.from_detector_error_model(dem_p_z)
    dem_m_z = circuit_m_z.detector_error_model(decompose_errors=True)
    m_m_z = pymatching.Matching.from_detector_error_model(dem_m_z)

    # Converting the measurement sample into DEM sample and continue as usual with decoding
    cvrtr_0_x = circuit_0_x.compile_m2d_converter()
    dets_ops_0_x = cvrtr_0_x.convert(measurements = rstls_smpls_0_x, append_observables=True)
    num_dets = dem_0_x.num_detectors
    num_obs  = dem_0_x.num_observables
    dets_0_x = dets_ops_0_x[:, :num_dets]
    obs_0_x  = dets_ops_0_x[:, num_dets:num_dets+num_obs]


    cvrtr_1_x = circuit_1_x.compile_m2d_converter()
    dets_ops_1_x = cvrtr_1_x.convert(measurements = rstls_smpls_1_x, append_observables=True)
    num_dets = dem_1_x.num_detectors
    num_obs  = dem_1_x.num_observables
    dets_1_x = dets_ops_1_x[:, :num_dets]
    obs_1_x  = dets_ops_1_x[:, num_dets:num_dets+num_obs]


    cvrtr_p_z = circuit_p_z.compile_m2d_converter()
    dets_ops_p_z = cvrtr_p_z.convert(measurements = rstls_smpls_p_z, append_observables=True)
    num_dets = dem_p_z.num_detectors
    num_obs  = dem_p_z.num_observables
    dets_p_z = dets_ops_p_z[:, :num_dets]
    obs_p_z  = dets_ops_p_z[:, num_dets:num_dets+num_obs]


    cvrtr_m_z = circuit_m_z.compile_m2d_converter()
    dets_ops_m_z = cvrtr_m_z.convert(measurements = rstls_smpls_m_z, append_observables=True)
    num_dets = dem_m_z.num_detectors
    num_obs  = dem_m_z.num_observables
    dets_m_z = dets_ops_m_z[:, :num_dets]
    obs_m_z  = dets_ops_m_z[:, num_dets:num_dets+num_obs]

    # Decode all sample round in one Batch decode
    pred_0_x = m_0_x.decode_batch(dets_0_x)
    pred_1_x = m_1_x.decode_batch(dets_1_x)
    pred_p_z = m_p_z.decode_batch(dets_p_z)
    pred_m_z = m_m_z.decode_batch(dets_m_z)

    # Init final state List after EC
    final_meas_0_x : list = []
    final_meas_1_x : list = []
    final_meas_p_z : list = []
    final_meas_m_z : list = []

    # XOR flip with noiseless Measurement (looping over all samples and compoaring to the current logical outcome in the list)
    # -> Taking first entry for first logical observable
    for current_sample in range(samples):
        curr_meas_0_x = 1 - 2 * (pred_0_x[current_sample,0].astype(np.int8) ^ np.int8(log_state_0_x[current_sample]))
        curr_meas_1_x = 1 - 2 * (pred_1_x[current_sample,0].astype(np.int8) ^ np.int8(log_state_1_x[current_sample]))
        curr_meas_p_z = 1 - 2 * (pred_p_z[current_sample,0].astype(np.int8) ^ np.int8(log_state_p_z[current_sample]))
        curr_meas_m_z = 1 - 2 * (pred_m_z[current_sample,0].astype(np.int8) ^ np.int8(log_state_m_z[current_sample]))

        #Adding final XORed measurement into the list
        final_meas_0_x.append(curr_meas_0_x)
        final_meas_1_x.append(curr_meas_1_x)
        final_meas_p_z.append(curr_meas_p_z)
        final_meas_m_z.append(curr_meas_m_z)

    # Take the Average of each of them
    mu_x_pz = np.average(final_meas_0_x)
    mu_x_mz = np.average(final_meas_1_x)
    mu_z_px = np.average(final_meas_p_z)
    mu_z_mx = np.average(final_meas_m_z)

    # Calc PTM entries
    r_zx = 1/2 * (mu_z_px - mu_z_mx)
    r_xz = 1/2 * (mu_x_pz - mu_x_mz)

    # Create PTM Matrix
    ptm = [
        [r_xx, 0.0, r_xz], 
        [0.0, 0.0 , 0.0], 
        [r_zx, 0.0 , r_zz]]

    return(np.array(ptm))

In [6]:
building_ptm(circuit_0_x = c_0_x, circuit_1_x = c_1_x, circuit_0_z = c_0_z, circuit_1_z = c_1_z,
            circuit_p_x = c_p_x, circuit_m_x = c_m_x, circuit_p_z = c_p_z, circuit_m_z = c_m_z, rec_pos_log_x = rec_pos_log_x, 
            rec_pos_log_z = rec_pos_log_z, samples = 10_000)

IndexError: index 0 is out of bounds for axis 1 with size 0

In [30]:
rotated_surface_code(distance = 5, rounds = 6, log_obs = "Y", state_init = "+i")

10 49
11 51


stim.Circuit('''
    QUBIT_COORDS(0, 4) 0
    QUBIT_COORDS(0, 8) 1
    QUBIT_COORDS(1, 1) 2
    QUBIT_COORDS(1, 3) 3
    QUBIT_COORDS(1, 5) 4
    QUBIT_COORDS(1, 7) 5
    QUBIT_COORDS(1, 9) 6
    QUBIT_COORDS(2, 0) 7
    QUBIT_COORDS(2, 2) 8
    QUBIT_COORDS(2, 4) 9
    QUBIT_COORDS(2, 6) 10
    QUBIT_COORDS(2, 8) 11
    QUBIT_COORDS(3, 1) 12
    QUBIT_COORDS(3, 3) 13
    QUBIT_COORDS(3, 5) 14
    QUBIT_COORDS(3, 7) 15
    QUBIT_COORDS(3, 9) 16
    QUBIT_COORDS(4, 0) 17
    QUBIT_COORDS(4, 2) 18
    QUBIT_COORDS(4, 4) 19
    QUBIT_COORDS(4, 6) 20
    QUBIT_COORDS(4, 8) 21
    QUBIT_COORDS(4, 10) 22
    QUBIT_COORDS(5, 1) 23
    QUBIT_COORDS(5, 3) 24
    QUBIT_COORDS(5, 5) 25
    QUBIT_COORDS(5, 7) 26
    QUBIT_COORDS(5, 9) 27
    QUBIT_COORDS(6, 0) 28
    QUBIT_COORDS(6, 2) 29
    QUBIT_COORDS(6, 4) 30
    QUBIT_COORDS(6, 6) 31
    QUBIT_COORDS(6, 8) 32
    QUBIT_COORDS(7, 1) 33
    QUBIT_COORDS(7, 3) 34
    QUBIT_COORDS(7, 5) 35
    QUBIT_COORDS(7, 7) 36
    QUBIT_COORDS(7, 9) 37
    

In [ ]:
circuit = stim.Circuit('''
            R 1 3
            CX 0 1 2 3
           CX 4 3 2 1
            M 1 3
        ''')
circuit.solve_flow_measurements([
        stim.Flow("1 -> Z0*Z2"),
        stim.Flow("1 -> Z2*Z4"),
        stim.Flow("Z0*Z2 -> 1"),
        ])

[[0], [1], [0]]